# UAV Detection Benchmark — Colab Runner

Runs the full benchmark on Colab's free T4 GPU.
No notebook-style coding — just shell commands.

**Run cells 1-5 in order. If Colab disconnects, run cells 1, 2, 3, then cell 7 to resume.**

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Clone repo + install dependencies
!git clone https://github.com/YOUR_USERNAME/uav-benchmark.git /content/uav-benchmark 2>/dev/null || echo 'Repo already cloned'

# Colab already has torch + torchvision with CUDA pre-installed.
# Installing them from requirements.txt would pull PyPI CPU versions
# that break CUDA support. So we skip torch/torchvision and install the rest.
!cd /content/uav-benchmark && grep -v '^torch' requirements.txt | pip install -q -r /dev/stdin

# Verify GPU + dependencies
!nvidia-smi
!python -c "import torch; print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"NONE\"}')"
!python -c "import ultralytics; print(f'Ultralytics: {ultralytics.__version__}')"

In [ ]:
# Cell 3: Link dataset from Google Drive (symlinks — zero copy, instant)
import os

data_dir = '/content/uav-benchmark/data'
drive_images = '/content/drive/MyDrive/images'
drive_labels = '/content/drive/MyDrive/labels'

# Remove existing links/dirs if re-running
!rm -f {data_dir}/images {data_dir}/labels

# Create symlinks (no disk space used, reads directly from Drive)
os.symlink(drive_images, f'{data_dir}/images')
os.symlink(drive_labels, f'{data_dir}/labels')

# Verify structure
!echo '--- Images ---' && ls {data_dir}/images/
!echo '--- Labels ---' && ls {data_dir}/labels/
!echo '--- Train sample ---' && ls {data_dir}/images/train/ | head -5
!echo '--- Train count ---' && ls {data_dir}/images/train/ | wc -l

In [ ]:
# Cell 4: Verify everything (GPU + dataset + configs)
!cd /content/uav-benchmark && python src/benchmark.py --verify-only

In [ ]:
# Cell 5: Run full benchmark (Round 1 — model comparison)
# This takes ~6-8 hours on T4. Keep the browser tab open.
!cd /content/uav-benchmark && python src/benchmark.py

In [ ]:
# Cell 6: Save results to Google Drive
# Run this after benchmark completes OR periodically as a backup.
!mkdir -p /content/drive/MyDrive/uav-benchmark-results
!cp -r /content/uav-benchmark/runs /content/drive/MyDrive/uav-benchmark-results/
!cp -r /content/uav-benchmark/reports /content/drive/MyDrive/uav-benchmark-results/
!echo 'Results saved to Google Drive: uav-benchmark-results/'

---
## Recovery (if Colab disconnected)

If Colab disconnected mid-training:
1. Run **Cell 1** (mount Drive)
2. Run **Cell 2** (clone + install)
3. Run **Cell 3** (link dataset)
4. Run **Cell 7** below (restore + resume)

In [ ]:
# Cell 7: Restore previous runs + resume benchmark
# Only use this if Colab disconnected mid-training.

# Restore saved runs from Drive
!cp -r /content/drive/MyDrive/uav-benchmark-results/runs/* /content/uav-benchmark/runs/ 2>/dev/null || echo 'No previous runs to restore'

# Show which experiments already have weights
!echo '--- Completed experiments ---'
!ls /content/uav-benchmark/runs/*/weights/best.pt 2>/dev/null || echo 'None'

# Resume from the next unfinished experiment
# Change the name below to the first experiment WITHOUT weights above
# Options: yolo11s_baseline, yolo11s_p2, yolo26s, yolo26n
!cd /content/uav-benchmark && python src/benchmark.py --start-from yolo11s_p2

---
## Auto-backup (optional)

Run Cell 8 to automatically save results to Drive after each experiment.
This runs in the background — start it before Cell 5.

In [ ]:
# Cell 8: Auto-backup loop (run BEFORE Cell 5, in a separate cell)
# Checks every 10 minutes and copies any new results to Drive.
import threading
import time
import subprocess

# Create backup directory first
subprocess.run('mkdir -p /content/drive/MyDrive/uav-benchmark-results', shell=True)

def auto_backup():
    while True:
        time.sleep(600)  # every 10 minutes
        subprocess.run(
            'cp -r /content/uav-benchmark/runs /content/drive/MyDrive/uav-benchmark-results/ 2>/dev/null',
            shell=True
        )
        print(f'[Auto-backup] Saved at {time.strftime("%H:%M:%S")}')

backup_thread = threading.Thread(target=auto_backup, daemon=True)
backup_thread.start()
print('Auto-backup started (every 10 minutes)')